# DocMind · 01 — Fine-tune the reranker

Trains DocMind's **cross-encoder reranker**: given a question and one chunk of a document, it scores
whether that chunk answers the question. DocMind's hybrid retriever (bge-small + BM25) finds the right
chunk in its top 5 for 93.5% of gold-set questions but ranks it **first only 67%** of the time; the
reranker re-orders the top ~30 candidates so the answering chunk comes first.

**Runtime:** `Runtime → Change runtime type → T4 GPU`. Roughly 30–45 minutes end to end.

**Output:** `docmind-reranker-onnx.zip`, containing two ONNX models with their tokenizers:
- `reranker-finetuned/` — the fine-tuned DocMind reranker
- `reranker-base/` — the untouched base model, exported the same way, so the local gold-set eval can
  measure exactly what fine-tuning added

Unzip it into the repo's `models/` folder (`models/reranker-finetuned`, `models/reranker-base`).

### Why SQuAD 2.0 with in-document negatives
DocMind always searches **inside one uploaded document**, so the reranker's real job is to separate the
answering passage from *other passages of the same document*. The training pairs mirror that:
- **positive:** a question and the paragraph that answers it
- **hard negatives:** other paragraphs **from the same Wikipedia article**, chosen by BM25 so they share
  the question's vocabulary
- **"on-topic but unanswerable" negatives:** SQuAD 2.0's unanswerable questions paired with the paragraph
  they were written against — this teaches the score to mean *answers the question*, not *is about the
  topic*, which also helps DocMind refuse when nothing in the document answers.

The base model was trained on web search (MS MARCO), where negatives come from other websites;
in-document negatives are the domain shift this fine-tune targets. None of DocMind's gold-set documents
are used here — they stay test-only.

In [ ]:
!pip install -q "sentence-transformers==3.4.1" "datasets>=2.19,<4" "rank_bm25==0.2.2" onnx onnxruntime

In [ ]:
import collections, json, os, random, re, shutil

import numpy as np
import torch

BASE_MODEL     = "cross-encoder/ms-marco-MiniLM-L-12-v2"   # 33M params, Apache-2.0
MAX_LENGTH     = 384      # DocMind chunks are <= ~310 bge tokens; questions add ~20
NEG_PER_QUESTION = 3      # hard negatives per question
MAX_TRAIN_QUESTIONS = 40_000
DEV_QUERIES    = 2_000
EPOCHS         = 1
BATCH_SIZE     = 32
LEARNING_RATE  = 2e-5
SEED           = 42
OUT_DIR        = "/content/docmind-reranker"
EXPORT_DIR     = "/content/export"

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
assert torch.cuda.is_available(), "Switch the runtime to a GPU (T4) first."
print(torch.cuda.get_device_name(0))

## 1 · Build in-document training pairs

In [ ]:
from datasets import load_dataset
from rank_bm25 import BM25Okapi

squad = load_dataset("rajpurkar/squad_v2")

def tokenize(text):
    return re.findall(r"[a-z0-9]+", text.lower())

def group_by_article(split):
    articles = collections.defaultdict(list)
    for ex in split:
        articles[ex["title"]].append(ex)
    return articles

def ranked_other_paragraphs(question, own_context, contexts, bm25):
    scores = bm25.get_scores(tokenize(question))
    order = np.argsort(-scores)
    return [contexts[i] for i in order if contexts[i] != own_context]

def build(split, max_questions, for_eval=False):
    # Eval keeps only answerable questions (there must be a positive to rank).
    articles = group_by_article(split)
    questions = [ex for exs in articles.values() for ex in exs if ex["answers"]["text"] or not for_eval]
    random.shuffle(questions)
    bm25_cache = {}
    pairs, groups, used = [], [], 0
    for ex in questions:
        if used >= max_questions:
            break
        title = ex["title"]
        if title not in bm25_cache:
            contexts = list(dict.fromkeys(e["context"] for e in articles[title]))
            bm25_cache[title] = (contexts, BM25Okapi([tokenize(c) for c in contexts]))
        contexts, bm25 = bm25_cache[title]
        if len(contexts) < 2:
            continue
        used += 1
        others = ranked_other_paragraphs(ex["question"], ex["context"], contexts, bm25)
        if for_eval:
            groups.append({"query": ex["question"], "positive": [ex["context"]], "negative": others[:29]})
        else:
            hard = random.sample(others[:NEG_PER_QUESTION * 2], min(NEG_PER_QUESTION, len(others)))
            if ex["answers"]["text"]:
                pairs.append((ex["question"], ex["context"], 1.0))
                negatives = hard
            else:
                # on-topic paragraph that does NOT answer the question
                negatives = [ex["context"]] + hard[:NEG_PER_QUESTION - 1]
            pairs += [(ex["question"], n, 0.0) for n in negatives]
    return groups if for_eval else pairs

train_pairs = build(squad["train"], MAX_TRAIN_QUESTIONS)
dev_groups  = build(squad["validation"], DEV_QUERIES, for_eval=True)
labels = collections.Counter(l for _, _, l in train_pairs)
print(f"train pairs: {len(train_pairs):,}  (positives {int(labels[1.0]):,}, negatives {int(labels[0.0]):,})")
print(f"dev rerank queries: {len(dev_groups):,} (each: 1 positive vs up to 29 same-article paragraphs)")

## 2 · Baseline score, then fine-tune

The dev metric is **MRR@10 for re-ranking one answering paragraph against up to 29 other paragraphs of the
same article** — the same situation DocMind is in. It is measured on SQuAD 2.0 *validation* articles,
which never appear in training.

In [ ]:
from sentence_transformers import CrossEncoder, InputExample
from sentence_transformers.cross_encoder.evaluation import CERerankingEvaluator
from torch.utils.data import DataLoader

evaluator = CERerankingEvaluator(dev_groups, at_k=10, name="squad2-in-document")

model = CrossEncoder(BASE_MODEL, num_labels=1, max_length=MAX_LENGTH)
base_mrr = evaluator(model)
print(f"BASE model MRR@10: {base_mrr:.4f}")

In [ ]:
train_examples = [InputExample(texts=[q, p], label=l) for q, p, l in train_pairs]
loader = DataLoader(train_examples, shuffle=True, batch_size=BATCH_SIZE)

model.fit(
    train_dataloader=loader,
    evaluator=evaluator,
    epochs=EPOCHS,
    evaluation_steps=max(len(loader) // 4, 1),
    warmup_steps=int(0.1 * len(loader)),
    optimizer_params={"lr": LEARNING_RATE},
    output_path=OUT_DIR,
    save_best_model=True,
    use_amp=True,
    show_progress_bar=True,
)

best = CrossEncoder(OUT_DIR, num_labels=1, max_length=MAX_LENGTH)
tuned_mrr = evaluator(best)
print(f"BASE MRR@10:       {base_mrr:.4f}")
print(f"FINE-TUNED MRR@10: {tuned_mrr:.4f}   (delta {tuned_mrr - base_mrr:+.4f})")

## 3 · Export both models to ONNX

DocMind runs the reranker with `onnxruntime` + `tokenizers` (no PyTorch at serving time), matching how
the embedder already runs through fastembed. Each export is checked against the PyTorch model's logits.

In [ ]:
import onnxruntime as ort
from transformers import AutoModelForSequenceClassification, AutoTokenizer

def export_onnx(src, dst, label):
    model = AutoModelForSequenceClassification.from_pretrained(src).eval()
    tok = AutoTokenizer.from_pretrained(src)
    os.makedirs(dst, exist_ok=True)
    enc = tok(["What batch size was used?", "Who founded Unity?"],
              ["Batch size was set to 32, and weight decay was applied.", "The engine supports C#."],
              padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors="pt")
    names = [k for k in ("input_ids", "attention_mask", "token_type_ids") if k in enc]
    torch.onnx.export(
        model, tuple(enc[k] for k in names), f"{dst}/model.onnx",
        input_names=names, output_names=["logits"],
        dynamic_axes={**{k: {0: "batch", 1: "sequence"} for k in names}, "logits": {0: "batch"}},
        opset_version=17,
    )
    tok.save_pretrained(dst)   # writes tokenizer.json (fast tokenizer) used by DocMind
    with open(f"{dst}/docmind_reranker.json", "w") as f:
        json.dump({"label": label, "source": src, "base_model": BASE_MODEL, "max_length": MAX_LENGTH,
                   "input_names": names}, f, indent=2)

    session = ort.InferenceSession(f"{dst}/model.onnx", providers=["CPUExecutionProvider"])
    onnx_logits = session.run(None, {k: enc[k].numpy() for k in names})[0]
    with torch.no_grad():
        torch_logits = model(**{k: enc[k] for k in names}).logits.numpy()
    assert np.allclose(onnx_logits, torch_logits, atol=1e-4), (onnx_logits, torch_logits)
    print(f"{label}: exported to {dst}, ONNX matches PyTorch, logits={onnx_logits.ravel().round(3)}")

shutil.rmtree(EXPORT_DIR, ignore_errors=True)
export_onnx(BASE_MODEL, f"{EXPORT_DIR}/reranker-base", "base")
export_onnx(OUT_DIR, f"{EXPORT_DIR}/reranker-finetuned", "finetuned")
with open(f"{EXPORT_DIR}/training_summary.json", "w") as f:
    json.dump({"base_model": BASE_MODEL, "train_pairs": len(train_pairs), "epochs": EPOCHS,
               "squad2_in_document_mrr10": {"base": base_mrr, "finetuned": tuned_mrr}}, f, indent=2)

In [ ]:
archive = shutil.make_archive("/content/docmind-reranker-onnx", "zip", EXPORT_DIR)
print(archive, f"{os.path.getsize(archive) / 1e6:.0f} MB")
from google.colab import files
files.download(archive)